In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Charger NSL-KDD
columns = ['duration','protocol_type','service','flag','src_bytes','dst_bytes',
           'land','wrong_fragment','urgent','hot','num_failed_logins','logged_in',
           'num_compromised','root_shell','su_attempted','num_root','num_file_creations',
           'num_shells','num_access_files','num_outbound_cmds','is_host_login',
           'is_guest_login','count','srv_count','serror_rate','srv_serror_rate',
           'rerror_rate','srv_rerror_rate','same_srv_rate','diff_srv_rate',
           'srv_diff_host_rate','dst_host_count','dst_host_srv_count',
           'dst_host_same_srv_rate','dst_host_diff_srv_rate',
           'dst_host_same_src_port_rate','dst_host_srv_diff_host_rate',
           'dst_host_serror_rate','dst_host_srv_serror_rate','dst_host_rerror_rate',
           'dst_host_srv_rerror_rate','label','difficulty']

df = pd.read_csv('../data/raw/KDDTrain+.txt', names=columns)

# Encoder les colonnes catégorielles
le = LabelEncoder()
for col in ['protocol_type', 'service', 'flag']:
    df[col] = le.fit_transform(df[col])

# Binariser le label (normal=0, attaque=1)
df['is_attack'] = (df['label'] != 'normal').astype(int)

# Features numériques uniquement
X = df.drop(['label', 'difficulty', 'is_attack'], axis=1)
y = df['is_attack']

# Normaliser
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Sous-échantillon pour les tests rapides
X_sample = X_scaled[:5000]
y_sample = y[:5000]
print(f"Dataset prêt : {X_sample.shape}")

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, confusion_matrix
import time

print("=== Isolation Forest ===")
t0 = time.time()

iso = IsolationForest(contamination=0.3, random_state=42, n_jobs=-1)
iso_preds = iso.fit_predict(X_sample)

# IsolationForest retourne -1 (anomalie) et 1 (normal)
# On convertit en 0/1
iso_binary = (iso_preds == -1).astype(int)

print(f"Temps d'entraînement: {time.time()-t0:.2f}s")
print(classification_report(y_sample, iso_binary, target_names=['Normal','Attaque']))

import seaborn as sns
import matplotlib.pyplot as plt
cm = confusion_matrix(y_sample, iso_binary)
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds',
            xticklabels=['Normal','Attaque'], yticklabels=['Normal','Attaque'])
plt.title("Isolation Forest — Confusion Matrix")
plt.tight_layout()
plt.savefig('../data/isolation_forest_cm.png')
plt.show()

In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.decomposition import PCA

print("=== DBSCAN ===")

# Réduire la dimension pour visualisation
pca = PCA(n_components=2)
X_2d = pca.fit_transform(X_sample[:2000])
y_2d = y_sample[:2000]

t0 = time.time()
dbscan = DBSCAN(eps=0.5, min_samples=5, n_jobs=-1)
clusters = dbscan.fit_predict(X_2d)
print(f"Temps: {time.time()-t0:.2f}s")
print(f"Clusters trouvés: {len(set(clusters)) - (1 if -1 in clusters else 0)}")
print(f"Points outliers (bruit): {(clusters == -1).sum()}")

# Visualisation
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.scatter(X_2d[:, 0], X_2d[:, 1], c=clusters, cmap='tab10', s=5, alpha=0.7)
plt.title("DBSCAN — Clusters trouvés")
plt.colorbar()

plt.subplot(1, 2, 2)
plt.scatter(X_2d[:, 0], X_2d[:, 1], c=y_2d, cmap='RdYlGn', s=5, alpha=0.7)
plt.title("Vraies labels (vert=normal, rouge=attaque)")
plt.tight_layout()
plt.savefig('../data/dbscan_clusters.png')
plt.show()

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score

print("=== Random Forest ===")

X_train, X_test, y_train, y_test = train_test_split(
    X_sample, y_sample, test_size=0.2, random_state=42
)

t0 = time.time()
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
print(f"Temps entraînement: {time.time()-t0:.2f}s")

y_pred = rf.predict(X_test)
print(classification_report(y_test, y_pred, target_names=['Normal','Attaque']))
print(f"F1-score: {f1_score(y_test, y_pred):.3f}")

# Feature importance
feature_names = df.drop(['label','difficulty','is_attack'], axis=1).columns
importances = pd.Series(rf.feature_importances_, index=feature_names)
top_features = importances.nlargest(15)

plt.figure(figsize=(10,6))
top_features.plot(kind='barh', color='steelblue')
plt.title("Top 15 Features — Random Forest")
plt.tight_layout()
plt.savefig('../data/rf_feature_importance.png')
plt.show()

In [ ]:
# Tableau comparatif final
results = {
    'Algorithme': ['Isolation Forest', 'DBSCAN', 'Random Forest'],
    'Type': ['Non supervisé', 'Non supervisé (clustering)', 'Supervisé'],
    'Avantages': [
        'Pas besoin de labels, rapide',
        'Découverte de patterns, visualisation',
        'Haute précision si labels disponibles'
    ],
    'Inconvénients': [
        'Moins précis, pas de classes spécifiques',
        'Sensible aux paramètres eps/min_samples',
        'Nécessite des données labellisées'
    ],
    'Usage dans RootTrap': [
        'Détection anomalies logs honeypot bruts (P2)',
        'Profiling comportemental attaquants',
        'Classification ATT&CK (modèle principal)'
    ]
}

df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))
df_results.to_csv('../data/algo_comparison.csv', index=False)